In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)


# 📚 第三周-Day4：DPO与其他对齐方法对比

> **RLHF 很强大，但它就像一架需要四台发动机同时运转的飞机——昂贵、复杂、还容易出故障。2023年，斯坦福团队提出了一个石破天惊的问题：如果数学告诉我们奖励函数可以被完全消去呢？DPO就此诞生，它把对齐训练从"造火箭"变成了"骑自行车"。**


## 📅 学习进度

| 阶段 | 状态 |
|------|------|
| W1：Transformer 基础架构 | ✅ 已完成 |
| W2：Transformer 深入理解 | ✅ 已完成 |
| **W3：大模型训练全景** | 🔄 **进行中（Day 4/7）** |
| W4-W12 | ⏳ 待开始 |

---


## 一、为什么需要 DPO？

### 1.1 RLHF 的四大痛点

昨天我们学了 RLHF，看起来很完美。但工业界落地时，踩了一堆坑：

**🔴 痛点1：要同时加载 4 个模型！**


In [ ]:
RLHF 显存需求：
  ① 正在训练的策略模型 (Policy Model)
  ② 冻结的参考模型 (Reference Model)
  ③ 奖励模型 (Reward Model)
  ④ SFT 模型（有时用作参考）
  
至少需要 4× A100 (80GB) 才能跑得动！


**🔴 痛点2：PPO 超级不稳定**


In [ ]:
学习率稍大 → 模型输出乱码
clip ratio不对 → 训练震荡
KL系数太小 → 模型"走火入魔"
KL系数太大 → 模型不学习


**🔴 痛点3：计算成本爆炸**
- 奖励模型需要单独训练
- PPO 需要大量采样（on-policy）
- 总训练时间是 SFT 的 5-10 倍

**🔴 痛点4：调试噩梦**
- PPO 有 7-10 个超参数需要调
- 每个超参数互相影响
- 训练不收敛时，不知道是哪个参数的问题

### 1.2 DPO 的革命性简化

**2023年，斯坦福团队发表了一篇震动 AI 界的论文：**

> "我们证明了 RLHF 中的强化学习循环可以被简化为一个简单的分类问题。"

**核心发现**：RLHF 的优化目标存在**闭式解（closed-form solution）**——也就是说，你可以直接算出最优解，而不需要一步步迭代！

**RLHF vs DPO 流程对比：**


In [ ]:
RLHF（复杂版）:
  偏好数据 → 训练奖励模型 → 用PPO循环优化 → 反复迭代
  
DPO（简化版）:
  偏好数据 → 直接训练LLM（一个损失函数搞定）


**生活类比**：


In [ ]:
RLHF = 先请美食评委给每道菜打分 → 再根据评分改进菜谱 → 反复尝试
DPO  = 直接告诉厨师"A菜比B菜好吃" → 厨师自己领悟 → 一次搞定


---


## 二、核心原理详解

### 2.1 DPO 的数学直觉

别怕数学，我们一步步来。

**Step 1: RLHF 的原始目标**

RLHF 想最大化的是：


In [ ]:
目标 = 奖励(r(x,y)) - β × KL(π || π_ref)

说人话：最大化奖励分数，同时不要偏离原始模型太远


**Step 2: 数学魔法——解方程**

斯坦福团队发现：这个优化问题可以**解析求解**！

最优的策略模型满足：


In [ ]:
π*(y|x) = π_ref(y|x) × exp(r(x,y) / β) / Z(x)


**关键洞察**：奖励函数 r(x,y) 可以用策略模型 π 来表达！


In [ ]:
r(x,y) = β × log(π(y|x) / π_ref(y|x)) + β × log(Z(x))


**Step 3: 代入偏好对，得到 DPO Loss**

对于偏好对 (好回答 y_w, 坏回答 y_l)：


In [ ]:
DPO Loss = -log(σ(β × [log π(y_w|x)/π_ref(y_w|x) - log π(y_l|x)/π_ref(y_l|x)]))


**用一句话解释**：让模型对"好回答"的输出概率（相对于参考模型）**上升**，对"坏回答"的输出概率**下降**。

### 2.2 DPO 的关键超参数：β

**β（beta）** 是 DPO 唯一的重要超参数，它控制模型偏离参考模型的程度：

| β 值 | 行为 | 效果 |
|------|------|------|
| β 太小（0.01） | 模型激进改变 | 可能"过度优化"，输出不自然 |
| β 适中（0.1-0.5） | 温和学习 | 最常用，效果好且稳定 |
| β 太大（1.0+） | 模型几乎不变 | 学习太慢，效果不明显 |

**生活类比**：β 就像学习强度的"旋钮"🎛️
- β小 = 把油门踩到底，学得快但容易翻车
- β大 = 慢慢学，稳但可能学不够

### 2.3 DPO vs RLHF 全面对比

| 维度 | RLHF | DPO |
|------|------|-----|
| **需要奖励模型？** | ✅ 必须单独训练 | ❌ 不需要 |
| **需要PPO？** | ✅ 必须用PPO循环 | ❌ 不需要 |
| **同时加载模型数** | 4个 | 2个（当前+参考） |
| **显存需求** | ≥80GB | ~24GB |
| **训练稳定性** | ⚠️ 不稳定 | ✅ 非常稳定 |
| **超参数数量** | 7-10个 | 2-3个 |
| **训练速度** | 慢 | 快3-5倍 |
| **数据格式** | (问题, 回答, 分数) | (问题, 好回答, 坏回答) |
| **效果上限** | 理论上更高 | 略低但差距缩小 |

### 2.4 其他对齐方法概览

除了 DPO，研究界还提出了多种变体：

#### 🔵 KTO（Kahneman-Tversky Optimization）
- **灵感**：诺贝尔奖得主卡尼曼的**前景理论**——人类对损失比对获得更敏感
- **数据格式**：只需要"好/坏"标签，**不需要成对比较**
- **优势**：数据标注成本降低 60%+
- **适合场景**：有大量点赞/踩的数据（如客服评价）

#### 🟡 GRPO（Group Relative Policy Optimization）
- **来源**：DeepSeek 团队提出，用在 DeepSeek-Math 和 R1 上
- **核心思想**：对同一问题生成一组回答，用组内排名来优化
- **优势**：完全不需要奖励模型，靠"群体比较"学习
- **适合场景**：提升推理能力（DeepSeek R1 就是这么训的）

#### 🟢 CDPO（Contrastive DPO）
- 在 DPO 基础上增加对比学习，增强区分度
- 适合细粒度的偏好对齐

#### 🟠 IPO（Identity Preference Optimization）
- 修正了 DPO 在大数据集上可能过拟合的问题
- 数学上更严谨，但实际效果提升有限

### 2.5 选型决策树


In [ ]:
你的预算和资源如何？
├── 💰 充足（多GPU + 大团队）
│   └── 追求极致效果？ → RLHF
│   └── 快速迭代？ → DPO + 定期 RLHF 微调
├── 💵 中等（1-2张GPU）
│   └── 有成对偏好数据？ → DPO ✅（首选！）
│   └── 只有好坏标签？ → KTO
│   └── 提升推理能力？ → GRPO
└── 🪙 有限（单卡4060Ti）
    └── DPO + LoRA
    └── 或 GRPO 试试推理提升


---


## 三、代码实战

### 3.1 DPO 损失函数实现


In [ ]:
import numpy as np

def dpo_loss(log_prob_win, log_prob_lose, 
             log_prob_win_ref, log_prob_lose_ref, beta=0.1):
    """
    DPO 损失函数实现
    
    参数:
        log_prob_win:  当前模型对好回答的对数概率
        log_prob_lose: 当前模型对坏回答的对数概率
        log_prob_win_ref:  参考模型对好回答的对数概率
        log_prob_lose_ref: 参考模型对坏回答的对数概率
        beta: KL惩罚系数（默认0.1）
    
    返回:
        DPO损失值（越小越好）
    """
    # 计算隐式奖励
    implicit_reward_win = beta * (log_prob_win - log_prob_win_ref)
    implicit_reward_lose = beta * (log_prob_lose - log_prob_lose_ref)
    
    # DPO Loss = -log(sigmoid(r_win - r_lose))
    logits = implicit_reward_win - implicit_reward_lose
    loss = -np.log(1 / (1 + np.exp(-logits)))
    
    return loss

# 测试
loss_good = dpo_loss(-2.0, -4.0, -3.0, -3.0, beta=0.1)
loss_bad = dpo_loss(-4.0, -2.0, -3.0, -3.0, beta=0.1)
loss_neutral = dpo_loss(-3.0, -3.0, -3.0, -3.0, beta=0.1)

print(f"模型判断正确时的Loss: {loss_good:.4f}（应该较低）")
print(f"模型判断错误时的Loss: {loss_bad:.4f}（应该较高）")
print(f"模型还没学到时的Loss:  {loss_neutral:.4f}（中间值）")


### 3.2 DPO 训练过程模拟


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 设置中文字体
from matplotlib import font_manager
font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

np.random.seed(42)
epochs = 50
beta = 0.2

# 模拟训练过程
model_quality = np.zeros(epochs)
train_loss = np.zeros(epochs)
accuracy = np.zeros(epochs)

for epoch in range(epochs):
    # 模型质量逐渐提升
    model_quality[epoch] = 0.1 * (1 - np.exp(-epoch / 10)) + 0.01 * np.random.randn()
    
    # 隐式 logit 随质量提升而增大
    logit = model_quality[epoch] * 5
    
    # DPO Loss
    train_loss[epoch] = -np.log(1 / (1 + np.exp(-beta * logit))) + 0.02 * np.random.randn()
    
    # 偏好准确率
    accuracy[epoch] = 1 / (1 + np.exp(-logit)) + 0.02 * np.random.randn()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].plot(range(epochs), train_loss, 'b-', linewidth=2)
axes[0].set_title('DPO 训练 Loss', fontsize=14)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(range(epochs), accuracy, 'g-', linewidth=2)
axes[1].set_title('偏好准确率', fontsize=14)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_ylim(0.4, 1.0)
axes[1].grid(True, alpha=0.3)

axes[2].plot(range(epochs), model_quality, 'r-', linewidth=2)
axes[2].set_title('模型质量提升', fontsize=14)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Quality Score')
axes[2].grid(True, alpha=0.3)

plt.suptitle('DPO 训练全过程模拟', fontsize=16)
plt.tight_layout()
plt.show()
print("✅ Loss下降，准确率上升，训练非常稳定——这就是DPO的魅力！")


### 3.3 β 参数影响可视化


In [ ]:
betas = np.linspace(0.01, 1.0, 100)

# 不同场景下的Loss
logits_good = 2.0   # 模型正确区分
logits_bad = -2.0   # 模型判断错误
logits_neutral = 0.0  # 模型还没学会

loss_good = -np.log(1 / (1 + np.exp(-betas * logits_good)))
loss_bad = -np.log(1 / (1 + np.exp(-betas * logits_bad)))
loss_neutral = -np.log(1 / (1 + np.exp(-betas * logits_neutral)))

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(betas, loss_good, 'g-', linewidth=2, label='模型正确（好回答概率高）')
ax.plot(betas, loss_neutral, 'y-', linewidth=2, label='模型还没学会')
ax.plot(betas, loss_bad, 'r-', linewidth=2, label='模型判断错误')
ax.axvspan(0.1, 0.5, alpha=0.1, color='blue', label='推荐β范围')
ax.set_xlabel('β (beta)', fontsize=13)
ax.set_ylabel('DPO Loss', fontsize=13)
ax.set_title('β 参数对 DPO 训练的影响', fontsize=15)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---


## 四、可视化理解

### 4.1 DPO vs RLHF 雷达图


In [ ]:
categories = ['训练稳定性', '资源友好', '数据友好', '实现简单', 
              '调试容易', '效果上限']
rlhf_scores = [3, 2, 4, 2, 2, 9]
dpo_scores = [8, 7, 6, 8, 8, 7]

angles = np.linspace(0, 2 * np.pi, len(categories), endpoint=False).tolist()
rlhf_scores += rlhf_scores[:1]
dpo_scores += dpo_scores[:1]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
ax.plot(angles, rlhf_scores, 'r-o', linewidth=2, label='RLHF')
ax.fill(angles, rlhf_scores, alpha=0.15, color='red')
ax.plot(angles, dpo_scores, 'b-s', linewidth=2, label='DPO')
ax.fill(angles, dpo_scores, alpha=0.15, color='blue')

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=12)
ax.set_title('RLHF vs DPO 全面对比', fontsize=15, pad=20)
ax.legend(loc='upper right', fontsize=12)
plt.tight_layout()
plt.show()
print("DPO 在工程友好度上完胜，RLHF 在效果上限上略优。")


### 4.2 对齐方法生态图


In [ ]:
methods = ['RLHF', 'DPO', 'KTO', 'GRPO', 'IPO']
ease = [2, 8, 7, 5, 7]       # 实现容易度
power = [9, 7, 6, 8, 7]      # 效果上限
data_efficiency = [4, 6, 9, 8, 6]  # 数据效率

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['red', 'blue', 'green', 'orange', 'purple']

for i, m in enumerate(methods):
    ax.scatter(ease[i], power[i], s=data_efficiency[i]**2*30, 
               c=colors[i], alpha=0.7, edgecolors='black', label=m)
    ax.annotate(m, (ease[i], power[i]), fontsize=13, fontweight='bold',
                xytext=(8, 8), textcoords='offset points')

ax.set_xlabel('实现容易度', fontsize=13)
ax.set_ylabel('效果上限', fontsize=13)
ax.set_title('对齐方法生态图（圆圈大小 = 数据效率）', fontsize=15)
ax.legend(fontsize=12)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 11)
ax.set_ylim(4, 11)
plt.tight_layout()
plt.show()


---


## 五、业务关联

### 5.1 企业场景的对齐方法选择

| 场景 | 推荐方法 | 理由 |
|------|---------|------|
| 客服对话优化 | **DPO** | 稳定、高效、有明确好坏回答对比 |
| 内容安全过滤 | **KTO** | 只有好/坏标签，不需要成对比较 |
| 推理能力增强 | **GRPO** | DeepSeek R1 验证过的推理提升方案 |
| 追求极致效果 | RLHF | 效果上限最高，但成本也最高 |
| 小团队快速上线 | **DPO + LoRA** | 一张显卡就能跑 |

### 5.2 实际成本对比


In [ ]:
以 7B 模型对齐为例（估算）：

RLHF:
  奖励模型训练: $2,000
  PPO 训练:     $8,000
  超参调试:     $5,000
  总计:         ~$15,000

DPO:
  偏好数据准备: $1,000
  DPO 训练:     $1,500
  超参调试:     $500
  总计:         ~$3,000

💡 DPO 成本约为 RLHF 的 1/5


### 5.3 和 LangChat/Agent 的关系

- **LangChat**：DPO 微调后的模型在安全性和有用性上都有明显提升
- **Agent**：DPO 可以训练 Agent 的决策策略，让多步推理更符合人类期望
- **企业 AI**：DPO 是目前性价比最高的对齐方案，80%的场景都够用

---


## 六、常见误区

### ❌ 误区1："DPO 完全替代了 RLHF"
**事实**：DPO 是简化，不是替代。在追求极致效果的大公司（如 OpenAI），仍然会使用 RLHF。但中小团队和大多数应用场景，DPO 确实够用了。

### ❌ 误区2："DPO 不需要参考模型"
**事实**：DPO 仍然需要一个冻结的参考模型（通常是 SFT 模型）来计算概率差异。只是相比 RLHF 少了奖励模型和 PPO 的计算开销。

### ❌ 误区3："KTO 效果一定不如 DPO"
**事实**：当数据量充足时，KTO 的效果可以接近甚至匹配 DPO，且标注成本更低。关键看你的数据类型。

### ❌ 误区4："GRPO 只是 DPO 的变体"
**事实**：GRPO 有独立的算法设计——它通过组内排名来估计相对优势，不需要成对偏好数据，也不需要参考模型。它是为推理能力优化设计的。

---


## 🧪 课堂练习（5分钟）

**练习1**：用一句话解释 DPO 为什么不需要奖励模型。

**练习2**：如果你有以下数据，你会选择哪种对齐方法？


In [ ]:
数据：10000条用户评价（点赞👍或踩👎）
没有成对的"A比B好"标注


**练习3**：β 从 0.1 调到 0.01 会发生什么？从 0.1 调到 1.0 又会怎样？

---


## 📝 课后测试（15分钟）

**❶** DPO 的损失函数本质上是什么类型？
- A. 均方误差
- B. 二元交叉熵
- C. 对比损失
- D. Huber 损失

**❷** DPO 中 β 参数越大，模型会怎样？
- A. 越偏离参考模型
- B. 越保守（接近参考模型）
- C. 训练越快
- D. 没有影响

**❸** 以下哪个方法完全不需要成对偏好数据？
- A. DPO
- B. RLHF
- C. KTO
- D. CDPO

**❹** GRPO 是哪个团队提出的？主要用于什么？
- A. OpenAI，对话优化
- B. DeepSeek，推理能力
- C. Stanford，安全对齐
- D. Meta，多语言

**❺** 简答题：列举 DPO 相比 RLHF 的三个主要优势和一个主要劣势。

---


## 🔑 今日术语

| 英文 | 音标 | 中文 |
|------|------|------|
| DPO | /diː piː oʊ/ | 直接偏好优化 |
| KTO | /keɪ tiː oʊ/ | 前景理论优化 |
| GRPO | /dʒiː ɑːr piː oʊ/ | 群体相对策略优化 |
| Reward Hacking | /rɪˈwɔːrd ˈhækɪŋ/ | 奖励作弊 |
| Preference Data | /ˈprefrəns ˈdeɪtə/ | 偏好数据 |
| Contrastive Loss | /kənˈtræstɪv lɒs/ | 对比损失 |
| Closed-form Solution | /kloʊzd fɔːrm səˈluːʃən/ | 闭式解 |
| Policy Model | /ˈpɒləsi ˈmɒdl/ | 策略模型 |

---


## 📎 参考资源

### 必读论文
1. 📄 **Direct Preference Optimization: Your Language Model is Secretly a Reward Model** (Rafailov et al., 2023)
   - https://arxiv.org/abs/2305.18290
2. 📄 **KTO: Model Alignment as Prospect Theoretic Optimization** (Ethayarajh et al., 2024)
   - https://arxiv.org/abs/2402.01306
3. 📄 **DeepSeekMath / GRPO** (Shao et al., 2024)
   - https://arxiv.org/abs/2402.03300

### 视频推荐
1. 📺 **DPO 原理推导+实战**（B站，约25分钟）
   - https://www.bilibili.com/video/BV1m8FnzpEXm/
2. 📺 **RLHF 到 DPO 公式推导**（B站，约20分钟）
   - https://www.bilibili.com/video/BV1wfN6eWE8Z/

### 明日预告
明天我们把本周所有知识串成**完整训练流水线**——从数据准备到部署上线的 7 步走，加上超参数调优方法论！🚀
